# CheckSent-BN Sentiment Classification
**Model:** meta-llama/Llama-3.1-8B-Instruct  
**Strategy:** Few-shot

## 1. Import Libraries

In [1]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.4 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import gc
import re
import random

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("Libraries imported successfully.")
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

Libraries imported successfully.
PyTorch: 2.11.0+cu128
Transformers: 5.16.1
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


## 2. Configuration

In [3]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

DATA_DIR = (
    "/kaggle/input/datasets/"
    "taufiqzahantushar/checksent-bn/"
    "CheckSent-BN"
)

TRAIN_PATH = os.path.join(
    DATA_DIR,
    "train_data.tsv"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "test_data.tsv"
)

MAX_NEW_TOKENS = 16

LABELS = [
    "NEGATIVE",
    "NEUTRAL",
    "POSITIVE"
]

## 3. Load Dataset

In [4]:
train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t"
)

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain sentiment distribution:")
print(train_df["Sentiment"].value_counts())

print("\nTest sentiment distribution:")
print(test_df["Sentiment"].value_counts())

Train shape: (9254, 6)
Test shape: (2314, 6)

Train sentiment distribution:
Sentiment
NEGATIVE    4573
NEUTRAL     3187
POSITIVE    1494
Name: count, dtype: int64

Test sentiment distribution:
Sentiment
NEGATIVE    1123
NEUTRAL      825
POSITIVE     366
Name: count, dtype: int64


## 4. Data Validation

In [5]:
required_columns = [
    "ID",
    "Date",
    "News Portal",
    "Headline",
    "Claim",
    "Sentiment"
]

assert all(
    column in train_df.columns
    for column in required_columns
)

assert all(
    column in test_df.columns
    for column in required_columns
)

assert train_df["Headline"].isna().sum() == 0
assert test_df["Headline"].isna().sum() == 0

assert train_df["Sentiment"].isna().sum() == 0
assert test_df["Sentiment"].isna().sum() == 0

assert set(test_df["Sentiment"].unique()) == set(LABELS)

print("\nDataset validation passed.")


Dataset validation passed.


## 5. Hugging Face Token

In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("gptoss")

print("Hugging Face token loaded.")

Hugging Face token loaded.


## 6. Tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully.


## 7. 4-bit Quantization

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configured.")

4-bit quantization configured.


## 8. Load Model

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("Model loaded successfully.")
print("Model device:", model.device)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully.
Model device: cuda:0


## 9. Select Few-shot Examples

In [10]:
few_shot_examples = {}

for label in LABELS:

    example = (
        train_df[
            train_df["Sentiment"] == label
        ]
        .iloc[0]
    )

    few_shot_examples[label] = {
        "headline": example["Headline"],
        "label": label
    }


print("\nFew-shot examples selected:\n")

for label in LABELS:

    print(
        f"{label}:"
    )

    print(
        few_shot_examples[label]["headline"]
    )

    print()


Few-shot examples selected:

NEGATIVE:
বিএসএফের মদতে অনুপ্রবেশ: মমতা, বাংলাকে অশান্ত করতে কেন্দ্রের ছক

NEUTRAL:
দৈর্ঘ্য মাত্র ৪ ইঞ্চি, উপত্যকায় জঙ্গি দমনে সেনার মারণাস্ত্র ‘কালো ভ্রমর’

POSITIVE:
বেঙ্গালুরুকে হারিয়ে অঘটন মহামেডানের, ডার্বির আগে বাড়তি অক্সিজেন পেল মোহনবাগান



## 10. Sentiment Parser

In [11]:
def parse_sentiment(output):

    if output is None:
        return None

    output = output.strip().upper()

    output = output.replace("`", "")
    output = output.replace("*", "")
    output = output.replace("\n", " ")

    if output == "NEGATIVE":
        return "NEGATIVE"

    if output == "NEUTRAL":
        return "NEUTRAL"

    if output == "POSITIVE":
        return "POSITIVE"

    matches = []

    for label in LABELS:

        if re.search(
            rf"\b{label}\b",
            output
        ):
            matches.append(label)

    if len(matches) == 1:
        return matches[0]

    return None

## 11. Few-shot System Prompt

In [12]:
SYSTEM_PROMPT = """
You are performing Bengali sentiment classification.

Classify the sentiment of a Bengali news headline into exactly
ONE of these labels:

NEGATIVE:
The headline expresses negative sentiment, negative events,
criticism, conflict, loss, danger, failure, or other negative
emotional polarity.

NEUTRAL:
The headline is factual, informational, descriptive, or does
not clearly express positive or negative sentiment.

POSITIVE:
The headline expresses positive sentiment, success,
achievement, benefit, celebration, praise, or other positive
emotional polarity.

The examples provided by the user are demonstrations of the
classification task.

Use the examples to understand the relationship between a
headline and its sentiment label.

Return ONLY ONE label.

Allowed labels:
NEGATIVE
NEUTRAL
POSITIVE
""".strip()

## 12. Single Sample Prediction

In [13]:
def predict_sentiment(headline):

    user_prompt = f"""
Here are three labeled examples.

Example 1:

Headline:
{few_shot_examples["NEGATIVE"]["headline"]}

Label:
NEGATIVE


Example 2:

Headline:
{few_shot_examples["NEUTRAL"]["headline"]}

Label:
NEUTRAL


Example 3:

Headline:
{few_shot_examples["POSITIVE"]["headline"]}

Label:
POSITIVE


Now classify the following headline.

Headline:
{headline}

Final label:
""".strip()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        0,
        input_length:
    ]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    prediction = parse_sentiment(
        generated_text
    )

    return prediction, generated_text

## 13. Sample Test

In [14]:
sample_indices = [
    0,
    1,
    2,
    10,
    20
]

print("Sample predictions:\n")

for idx in sample_indices:

    headline = test_df.iloc[idx]["Headline"]
    actual = test_df.iloc[idx]["Sentiment"]

    prediction, raw_output = predict_sentiment(
        headline
    )

    print("=" * 80)
    print("Index:", idx)
    print("Headline:", headline)
    print("Actual:", actual)
    print("Prediction:", prediction)
    print("Raw output:", repr(raw_output))

Sample predictions:



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Index: 0
Headline: বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায় লাগাতার হুমকি পোস্টে আতঙ্ক
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE\n\nThe headline expresses negative sentiment, with words like "ব�'
Index: 1
Headline: নেপালে নদীতে পড়ল ভারতীয় পর্যটকবোঝাই বাস, মৃত অন্তত ১৪
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE\n\nThe headline expresses negative sentiment, as it mentions a tragic event involving'
Index: 2
Headline: পুজো হবে নির্বিঘ্নে, আশ্বাস ইউনুস সরকারের, মহালয়ার প্রস্তুতি বাংলাদেশে
Actual: NEUTRAL
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 10
Headline: প্রবল বর্ষণে বিপর্যস্ত মেঘালয়, হড়পা বান ও ধসে মৃত ১০ জন
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE\n\nThe headline expresses negative sentiment, as it mentions "বি�'
Index: 20
Headline: ইউটিউবে রোনাল্ডোর সাতকাহন, কোন সাত কারণে সিআর সেভেনের চ্যানেলে বুঁদ ভক্তরা?
Actual: POSITIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE\n\nThe headline mentions "সাতকাহন'


## 14. Full Few-shot Evaluation

In [15]:
few_shot_predictions = []
few_shot_raw_outputs = []

print("\nStarting Few-shot evaluation...\n")

for idx, headline in enumerate(
    tqdm(
        test_df["Headline"],
        total=len(test_df),
        desc="Few-shot"
    )
):

    try:

        prediction, raw_output = predict_sentiment(
            headline
        )

    except Exception as e:

        print(
            f"\nError at index {idx}: {e}"
        )

        prediction = None
        raw_output = ""

    few_shot_predictions.append(
        prediction
    )

    few_shot_raw_outputs.append(
        raw_output
    )

print("\nFew-shot inference completed.")

print(
    "Total predictions:",
    len(few_shot_predictions)
)

print(
    "Unknown predictions:",
    sum(
        p is None
        for p in few_shot_predictions
    )
)


Starting Few-shot evaluation...



Few-shot:   0%|          | 0/2314 [00:00<?, ?it/s]


Few-shot inference completed.
Total predictions: 2314
Unknown predictions: 19


## 15. Prepare Predictions

In [16]:
y_true = test_df["Sentiment"].tolist()

y_pred = [
    prediction
    if prediction in LABELS
    else "UNKNOWN"
    for prediction in few_shot_predictions
]

print("\nPrediction distribution:")
print(
    pd.Series(y_pred).value_counts()
)


Prediction distribution:
NEGATIVE    1721
POSITIVE     466
NEUTRAL      108
UNKNOWN       19
Name: count, dtype: int64


## 16. Calculate Metrics

In [17]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0
    )
)

## 17. Main Results

In [18]:
print("\n" + "=" * 60)
print("LLAMA 3.1 8B INSTRUCT — FEW-SHOT RESULTS")
print("=" * 60)

print(
    f"Accuracy : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision: {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall   : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"Macro F1 : {macro_f1:.4f} "
    f"({macro_f1 * 100:.2f}%)"
)

print(
    f"Unknown  : {sum(p is None for p in few_shot_predictions)}"
)

print(
    f"Total test samples: {len(y_true)}"
)


LLAMA 3.1 8B INSTRUCT — FEW-SHOT RESULTS
Accuracy : 0.6072 (60.72%)
Precision: 0.6772 (67.72%)
Recall   : 0.5641 (56.41%)
Macro F1 : 0.5032 (50.32%)
Unknown  : 19
Total test samples: 2314


## 18. Class-wise Results

In [19]:
print("\n" + "=" * 60)
print("CLASS-WISE RESULTS")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=LABELS,
        digits=4,
        zero_division=0
    )
)


CLASS-WISE RESULTS
              precision    recall  f1-score   support

    NEGATIVE     0.6293    0.9644    0.7616      1123
     NEUTRAL     0.9259    0.1212    0.2144       825
    POSITIVE     0.4764    0.6066    0.5337       366

   micro avg     0.6122    0.6072    0.6097      2314
   macro avg     0.6772    0.5641    0.5032      2314
weighted avg     0.7109    0.6072    0.5304      2314



## 19. Confusion Matrix

In [20]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

print("\nConfusion Matrix:")
display(cm_df)


Confusion Matrix:


,NEGATIVE,NEUTRAL,POSITIVE
NEGATIVE,1083,4,35
NEUTRAL,499,100,209
POSITIVE,139,4,222


## 20. Final Result Table

In [21]:
few_shot_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Macro F1"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        macro_f1
    ]
})

print("\nFinal Few-shot Results:")
display(few_shot_results)


Final Few-shot Results:


,Metric,Score
0,Accuracy,0.607174
1,Precision,0.677202
2,Recall,0.564050
3,Macro F1,0.503206
